## Setup

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
raw_file_path = Path(
    "Datasets",
    "CommercialRetailRentalAnalysis20260629170349.csv"
)

processed_folder_path = Path("Processed Data")

processed_file_path = processed_folder_path / "commercial_retail_rental_clean.csv"

print("Raw file:")
print(raw_file_path)

print("\nProcessed file:")
print(processed_file_path)

Raw file:
Datasets\CommercialRetailRentalAnalysis20260629170349.csv

Processed file:
Processed Data\commercial_retail_rental_clean.csv


In [3]:
if raw_file_path.exists():
    print("Rental dataset found.")
else:
    print("Rental dataset not found.")

Rental dataset found.


In [4]:
rental_raw = pd.read_csv(raw_file_path)

rental_raw.head()

,Postal District,Floor Level,Floor Area (SQM),25th Percentile ($ PSM),Median ($PSM),75th Percentile ($PSM),Reference Period
0,1,B1 & Below,30 & Below,69.89,70.20,112.70,2021Q1
1,1,B1 & Below,>30 - 100,65.45,87.61,98.41,2021Q1
2,1,B1 & Below,>100 - 300,50.33,57.70,87.20,2021Q1
3,1,Level 1,30 & Below,87.36,123.94,222.00,2021Q1
4,1,Level 1,>30 - 100,75.26,109.37,147.04,2021Q1


## Initial Checks and Data Exploration

In [5]:
print("Number of rows and columns:")
print(rental_raw.shape)

print("\nColumn names:")
print(rental_raw.columns.tolist())

print("\nData types:")
print(rental_raw.dtypes)

Number of rows and columns:
(4320, 7)

Column names:
['Postal District', 'Floor Level', 'Floor Area (SQM)', '25th Percentile ($ PSM)', 'Median ($PSM)', '75th Percentile ($PSM)', 'Reference Period']

Data types:
Postal District              int64
Floor Level                 object
Floor Area (SQM)            object
25th Percentile ($ PSM)    float64
Median ($PSM)               object
75th Percentile ($PSM)      object
Reference Period            object
dtype: object


In [6]:
print("Missing values in each column:")
print(rental_raw.isnull().sum())

print("\nNumber of exact duplicate rows:")
print(rental_raw.duplicated().sum())

Missing values in each column:
Postal District            0
Floor Level                0
Floor Area (SQM)           0
25th Percentile ($ PSM)    0
Median ($PSM)              0
75th Percentile ($PSM)     0
Reference Period           0
dtype: int64

Number of exact duplicate rows:
0


In [7]:
rental = rental_raw.copy()

rental.head()

,Postal District,Floor Level,Floor Area (SQM),25th Percentile ($ PSM),Median ($PSM),75th Percentile ($PSM),Reference Period
0,1,B1 & Below,30 & Below,69.89,70.20,112.70,2021Q1
1,1,B1 & Below,>30 - 100,65.45,87.61,98.41,2021Q1
2,1,B1 & Below,>100 - 300,50.33,57.70,87.20,2021Q1
3,1,Level 1,30 & Below,87.36,123.94,222.00,2021Q1
4,1,Level 1,>30 - 100,75.26,109.37,147.04,2021Q1


In [8]:
rental = rental.rename(columns={
    "Postal District": "postal_district_number",
    "Floor Level": "floor_level",
    "Floor Area (SQM)": "floor_area_band",
    "25th Percentile ($ PSM)": "rent_psm_25th",
    "Median ($PSM)": "rent_psm_median",
    "75th Percentile ($PSM)": "rent_psm_75th",
    "Reference Period": "reference_period"
})

rental.head()

,postal_district_number,floor_level,floor_area_band,rent_psm_25th,rent_psm_median,rent_psm_75th,reference_period
0,1,B1 & Below,30 & Below,69.89,70.20,112.70,2021Q1
1,1,B1 & Below,>30 - 100,65.45,87.61,98.41,2021Q1
2,1,B1 & Below,>100 - 300,50.33,57.70,87.20,2021Q1
3,1,Level 1,30 & Below,87.36,123.94,222.00,2021Q1
4,1,Level 1,>30 - 100,75.26,109.37,147.04,2021Q1


In [9]:
text_columns = [
    "floor_level",
    "floor_area_band",
    "reference_period"
]

for column in text_columns:
    rental[column] = rental[column].astype("string")
    rental[column] = rental[column].str.strip()

In [10]:
print("Floor levels:")
print(rental["floor_level"].unique())

print("\nFloor area bands:")
print(rental["floor_area_band"].unique())

print("\nReference periods:")
print(rental["reference_period"].unique())

Floor levels:
<StringArray>
['B1 & Below', 'Level 1', 'Level 2 & 3', 'Level 4 & Above']
Length: 4, dtype: string

Floor area bands:
<StringArray>
['30 & Below', '>30 - 100', '>100 - 300', '>300']
Length: 4, dtype: string

Reference periods:
<StringArray>
['2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3',
 '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2',
 '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1']
Length: 21, dtype: string


In [11]:
rent_columns = [
    "rent_psm_25th",
    "rent_psm_median",
    "rent_psm_75th"
]

for column in rent_columns:
    rental[column] = rental[column].astype("string")

    rental[column] = rental[column].str.replace(
        ",",
        "",
        regex=False
    )

    rental[column] = pd.to_numeric(
        rental[column],
        errors="coerce"
    )

In [12]:
print("Rental-price data types:")
print(rental[rent_columns].dtypes)

print("\nMissing values after conversion:")
print(rental[rent_columns].isnull().sum())

Rental-price data types:
rent_psm_25th      Float64
rent_psm_median    Float64
rent_psm_75th      Float64
dtype: object

Missing values after conversion:
rent_psm_25th      0
rent_psm_median    0
rent_psm_75th      0
dtype: int64


In [13]:
rental["postal_district_number"] = pd.to_numeric(
    rental["postal_district_number"],
    errors="coerce"
)

rental["postal_district_number"] = rental[
    "postal_district_number"
].astype("Int64")

In [14]:
rental["postal_district"] = (
    "D"
    + rental["postal_district_number"]
    .astype("string")
    .str.zfill(2)
)

In [15]:
district_code_check = rental[[
    "postal_district_number",
    "postal_district"
]].drop_duplicates()

district_code_check = district_code_check.sort_values(
    "postal_district_number"
)

display(district_code_check)

,postal_district_number,postal_district
0,1,D01
297,2,D02
424,3,D03
558,4,D04
721,5,D05
894,6,D06
1114,7,D07
1378,8,D08
1541,9,D09
1870,10,D10


## Formatting Data

In [16]:
district_names = {
    "D01": "Raffles Place, Cecil, Marina, People's Park",
    "D02": "Anson, Tanjong Pagar",
    "D03": "Queenstown, Tiong Bahru",
    "D04": "Telok Blangah, Harbourfront",
    "D05": "Pasir Panjang, Hong Leong Garden, Clementi New Town",
    "D06": "High Street, Beach Road (part)",
    "D07": "Middle Road, Golden Mile",
    "D08": "Little India",
    "D09": "Orchard, Cairnhill, River Valley",
    "D10": "Ardmore, Bukit Timah, Holland Road, Tanglin",
    "D11": "Watten Estate, Novena, Thomson",
    "D12": "Balestier, Toa Payoh, Serangoon",
    "D13": "Macpherson, Braddell",
    "D14": "Geylang, Eunos",
    "D15": "Katong, Joo Chiat, Amber Road",
    "D16": "Bedok, Upper East Coast, Eastwood, Kew Drive",
    "D17": "Loyang, Changi",
    "D18": "Tampines, Pasir Ris",
    "D19": "Serangoon Garden, Hougang, Punggol",
    "D20": "Bishan, Ang Mo Kio",
    "D21": "Upper Bukit Timah, Clementi Park, Ulu Pandan",
    "D22": "Jurong",
    "D23": "Hillview, Dairy Farm, Bukit Panjang, Choa Chu Kang",
    "D24": "Lim Chu Kang, Tengah",
    "D25": "Kranji, Woodgrove",
    "D26": "Upper Thomson, Springleaf",
    "D27": "Yishun, Sembawang",
    "D28": "Seletar"
}

In [17]:
rental["postal_district_name"] = rental[
    "postal_district"
].map(district_names)

rental["postal_district_label"] = (
    rental["postal_district"]
    + " / "
    + rental["postal_district_name"]
)

In [18]:
district_labels = rental[[
    "postal_district",
    "postal_district_name",
    "postal_district_label"
]].drop_duplicates()

district_labels = district_labels.sort_values(
    "postal_district"
)

display(district_labels)

,postal_district,postal_district_name,postal_district_label
0,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park"
297,D02,"Anson, Tanjong Pagar","D02 / Anson, Tanjong Pagar"
424,D03,"Queenstown, Tiong Bahru","D03 / Queenstown, Tiong Bahru"
558,D04,"Telok Blangah, Harbourfront","D04 / Telok Blangah, Harbourfront"
721,D05,"Pasir Panjang, Hong Leong Garden, Clementi New...","D05 / Pasir Panjang, Hong Leong Garden, Clemen..."
894,D06,"High Street, Beach Road (part)","D06 / High Street, Beach Road (part)"
1114,D07,"Middle Road, Golden Mile","D07 / Middle Road, Golden Mile"
1378,D08,Little India,D08 / Little India
1541,D09,"Orchard, Cairnhill, River Valley","D09 / Orchard, Cairnhill, River Valley"
1870,D10,"Ardmore, Bukit Timah, Holland Road, Tanglin","D10 / Ardmore, Bukit Timah, Holland Road, Tanglin"


In [19]:
unmatched_districts = rental[
    rental["postal_district_name"].isnull()
]

print("Rows with unmatched postal districts:")
print(len(unmatched_districts))

Rows with unmatched postal districts:
0


In [20]:
rental["period_start"] = pd.PeriodIndex(
    rental["reference_period"],
    freq="Q-DEC"
).start_time

In [21]:
rental["year"] = rental["period_start"].dt.year

rental["quarter"] = (
    "Q"
    + rental["period_start"]
    .dt.quarter
    .astype(str)
)

In [22]:
period_check = rental[[
    "reference_period",
    "period_start",
    "year",
    "quarter"
]].drop_duplicates()

period_check = period_check.sort_values(
    "period_start"
)

display(period_check.head())

,reference_period,period_start,year,quarter
0,2021Q1,2021-01-01,2021,Q1
11,2021Q2,2021-04-01,2021,Q2
23,2021Q3,2021-07-01,2021,Q3
36,2021Q4,2021-10-01,2021,Q4
51,2022Q1,2022-01-01,2022,Q1


In [23]:
floor_level_order = [
    "B1 & Below",
    "Level 1",
    "Level 2 & 3",
    "Level 4 & Above"
]

rental["floor_level"] = pd.Categorical(
    rental["floor_level"],
    categories=floor_level_order,
    ordered=True
)

In [24]:
floor_area_order = [
    "30 & Below",
    ">30 - 100",
    ">100 - 300",
    ">300"
]

rental["floor_area_band"] = pd.Categorical(
    rental["floor_area_band"],
    categories=floor_area_order,
    ordered=True
)

In [25]:
column_order = [
    "postal_district_number",
    "postal_district",
    "postal_district_name",
    "postal_district_label",
    "floor_level",
    "floor_area_band",
    "rent_psm_25th",
    "rent_psm_median",
    "rent_psm_75th",
    "reference_period",
    "period_start",
    "year",
    "quarter"
]

rental = rental[column_order]

rental.head()

,postal_district_number,postal_district,postal_district_name,postal_district_label,floor_level,floor_area_band,rent_psm_25th,rent_psm_median,rent_psm_75th,reference_period,period_start,year,quarter
0,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",B1 & Below,30 & Below,69.89,70.2,112.7,2021Q1,2021-01-01,2021,Q1
1,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",B1 & Below,>30 - 100,65.45,87.61,98.41,2021Q1,2021-01-01,2021,Q1
2,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",B1 & Below,>100 - 300,50.33,57.7,87.2,2021Q1,2021-01-01,2021,Q1
3,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 1,30 & Below,87.36,123.94,222.0,2021Q1,2021-01-01,2021,Q1
4,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 1,>30 - 100,75.26,109.37,147.04,2021Q1,2021-01-01,2021,Q1


In [26]:
print("Cleaned dataset shape:")
print(rental.shape)

print("\nCleaned data types:")
print(rental.dtypes)

print("\nMissing values:")
print(rental.isnull().sum())

print("\nExact duplicate rows:")
print(rental.duplicated().sum())

Cleaned dataset shape:
(4320, 13)

Cleaned data types:
postal_district_number             Int64
postal_district           string[python]
postal_district_name              object
postal_district_label     string[python]
floor_level                     category
floor_area_band                 category
rent_psm_25th                    Float64
rent_psm_median                  Float64
rent_psm_75th                    Float64
reference_period          string[python]
period_start              datetime64[ns]
year                               int32
quarter                           object
dtype: object

Missing values:
postal_district_number    0
postal_district           0
postal_district_name      0
postal_district_label     0
floor_level               0
floor_area_band           0
rent_psm_25th             0
rent_psm_median           0
rent_psm_75th             0
reference_period          0
period_start              0
year                      0
quarter                   0
dtype: int64

Exa

In [27]:
rental_key_columns = [
    "postal_district",
    "floor_level",
    "floor_area_band",
    "reference_period"
]

duplicate_rental_keys = rental.duplicated(
    subset=rental_key_columns,
    keep=False
)

print("Rows with duplicated rental keys:")
print(duplicate_rental_keys.sum())

Rows with duplicated rental keys:
0


In [28]:
invalid_percentile_rows = rental[
    (rental["rent_psm_25th"] > rental["rent_psm_median"])
    |
    (rental["rent_psm_median"] > rental["rent_psm_75th"])
]

print("Rows with invalid percentile ordering:")
print(len(invalid_percentile_rows))

Rows with invalid percentile ordering:
0


In [29]:
invalid_districts = rental[
    rental["postal_district_number"].isnull()
    |
    ~rental["postal_district_number"].between(1, 28)
]

print("Rows with invalid postal districts:")
print(len(invalid_districts))

Rows with invalid postal districts:
0


In [30]:
expected_districts = set(range(1, 29))

available_districts = set(
    rental["postal_district_number"]
    .dropna()
    .astype(int)
)

missing_districts = sorted(
    expected_districts - available_districts
)

print("Postal districts missing from the rental dataset:")
print(missing_districts)

Postal districts missing from the rental dataset:
[24]


In [31]:
display(rental.head(10))

,postal_district_number,postal_district,postal_district_name,postal_district_label,floor_level,floor_area_band,rent_psm_25th,rent_psm_median,rent_psm_75th,reference_period,period_start,year,quarter
0,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",B1 & Below,30 & Below,69.89,70.2,112.7,2021Q1,2021-01-01,2021,Q1
1,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",B1 & Below,>30 - 100,65.45,87.61,98.41,2021Q1,2021-01-01,2021,Q1
2,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",B1 & Below,>100 - 300,50.33,57.7,87.2,2021Q1,2021-01-01,2021,Q1
3,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 1,30 & Below,87.36,123.94,222.0,2021Q1,2021-01-01,2021,Q1
4,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 1,>30 - 100,75.26,109.37,147.04,2021Q1,2021-01-01,2021,Q1
5,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 1,>100 - 300,84.43,119.72,160.18,2021Q1,2021-01-01,2021,Q1
6,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 2 & 3,30 & Below,82.13,100.0,119.72,2021Q1,2021-01-01,2021,Q1
7,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 2 & 3,>30 - 100,51.57,70.7,100.0,2021Q1,2021-01-01,2021,Q1
8,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 2 & 3,>100 - 300,46.28,67.49,92.07,2021Q1,2021-01-01,2021,Q1
9,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 2 & 3,>300,46.86,64.58,71.37,2021Q1,2021-01-01,2021,Q1


In [32]:
display(rental.tail(10))

,postal_district_number,postal_district,postal_district_name,postal_district_label,floor_level,floor_area_band,rent_psm_25th,rent_psm_median,rent_psm_75th,reference_period,period_start,year,quarter
4310,28,D28,Seletar,D28 / Seletar,Level 1,30 & Below,232.0,253.0,268.05,2023Q3,2023-07-01,2023,Q3
4311,28,D28,Seletar,D28 / Seletar,B1 & Below,>30 - 100,255.34,279.94,312.74,2023Q4,2023-10-01,2023,Q4
4312,28,D28,Seletar,D28 / Seletar,Level 1,>30 - 100,274.74,279.51,303.7,2023Q4,2023-10-01,2023,Q4
4313,28,D28,Seletar,D28 / Seletar,Level 2 & 3,>30 - 100,240.03,291.89,364.12,2023Q4,2023-10-01,2023,Q4
4314,28,D28,Seletar,D28 / Seletar,Level 2 & 3,>100 - 300,99.34,107.86,165.71,2023Q4,2023-10-01,2023,Q4
4315,28,D28,Seletar,D28 / Seletar,B1 & Below,>30 - 100,241.59,262.45,295.92,2024Q1,2024-01-01,2024,Q1
4316,28,D28,Seletar,D28 / Seletar,Level 1,>30 - 100,118.37,210.45,310.72,2024Q2,2024-04-01,2024,Q2
4317,28,D28,Seletar,D28 / Seletar,B1 & Below,>100 - 300,193.61,209.83,210.99,2024Q3,2024-07-01,2024,Q3
4318,28,D28,Seletar,D28 / Seletar,B1 & Below,30 & Below,179.79,305.78,437.78,2025Q2,2025-04-01,2025,Q2
4319,28,D28,Seletar,D28 / Seletar,B1 & Below,>30 - 100,173.6,205.93,279.18,2025Q4,2025-10-01,2025,Q4


In [33]:
processed_folder_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Processed Data folder is ready.")

Processed Data folder is ready.


In [34]:
rental.to_csv(
    processed_file_path,
    index=False
)

print("Cleaned rental dataset saved successfully.")
print(processed_file_path)

Cleaned rental dataset saved successfully.
Processed Data\commercial_retail_rental_clean.csv


In [35]:
if processed_file_path.exists():
    print("Saved file found.")
else:
    print("Saved file was not created.")

Saved file found.


## Final Cleaned Dataset

In [36]:
rental_saved_check = pd.read_csv(
    processed_file_path
)

print("Saved dataset shape:")
print(rental_saved_check.shape)

display(rental_saved_check.head())

Saved dataset shape:
(4320, 13)


,postal_district_number,postal_district,postal_district_name,postal_district_label,floor_level,floor_area_band,rent_psm_25th,rent_psm_median,rent_psm_75th,reference_period,period_start,year,quarter
0,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",B1 & Below,30 & Below,69.89,70.20,112.70,2021Q1,2021-01-01,2021,Q1
1,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",B1 & Below,>30 - 100,65.45,87.61,98.41,2021Q1,2021-01-01,2021,Q1
2,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",B1 & Below,>100 - 300,50.33,57.70,87.20,2021Q1,2021-01-01,2021,Q1
3,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 1,30 & Below,87.36,123.94,222.00,2021Q1,2021-01-01,2021,Q1
4,1,D01,"Raffles Place, Cecil, Marina, People's Park","D01 / Raffles Place, Cecil, Marina, People's Park",Level 1,>30 - 100,75.26,109.37,147.04,2021Q1,2021-01-01,2021,Q1
